# Практика · Відео: трекінг

> Лекція: [lecture.html](lecture.html) · Домашнє: [homework.md](homework.md) · Тест: [quiz.html](quiz.html)

> ⏱ Зошит нічого не навчає — тут немає жодної нейромережі. Заміряно: **близько 20 секунд**
> на чотирьох ядрах без відеокарти. Найдовше йде `cv2.TrackerMIL` (десятки мілісекунд
> на кадр) і густий оптичний потік.

Що зробимо:

1. Збудуємо синтетичне відео, у якому **істина відома за побудовою**: пʼять куль
   літають по полю 96×96 сорок кадрів, і ми знаємо точну рамку кожної на кожному кадрі.
2. Заміряємо **оптичний потік** — `calcOpticalFlowFarneback` і `calcOpticalFlowPyrLK` —
   на зсуві, який задали самі. Побачимо, де він точний до сотих пікселя, а де мовчки бреше.
3. Спробуємо вести рамку **самим потоком** і подивимось, як швидко вона тікає.
4. Зберемо **трекінг за детекцією**: детектор на кожному кадрі плюс зіставлення
   угорським алгоритмом по IoU.
5. Уведемо метрику **підміна номера** (ID switch) і зʼясуємо, звідки підміни беруться.
6. Перевіримо **модель руху** й **фільтр Калмана** — і покажемо, за яких умов
   вони починають щось важити.
7. Порахуємо **MOTA** з трьома доданками й побачимо, чого вона не бачить.
8. Заміряємо єдиний трекер, що лишився в OpenCV 5, — `cv2.TrackerMIL`.

Кожна точка міряється на **вісьмох зернах**, і поряд із середнім друкується розкид.
Різниця, менша за розкид, — не різниця.

In [ ]:
import sys, time
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

print("python  ", sys.version.split()[0])
print("cv2     ", cv2.__version__)
print("numpy   ", np.__version__)

# У OpenCV 5 модуля cv2.legacy більше немає — разом із ним пішли KCF, CSRT,
# MOSSE, MedianFlow і Boosting. Перевіряємо це, а не віримо памʼяті.
print("cv2.legacy є:", hasattr(cv2, "legacy"))
print("трекери, що лишились:", [n for n in dir(cv2) if n.startswith("Tracker") and n.endswith("_create")])

## 1 · Сцена, у якій істина відома за побудовою

Знімати справжнє відео нам нема з чого: пакетів `decord` і `av` у середовищі немає,
а тягнути файл із мережі зошит не має права. Але для цієї теми синтетика навіть краща:
**ми знаємо точну рамку кожної кулі на кожному кадрі**, тож будь-яку помилку трекера
можна порахувати, а не оцінити на око.

Пʼять куль радіуса 7 пікселів літають полем 96×96 і відбиваються від країв.
Сорок кадрів. Це наскрізний приклад усієї теми.

In [ ]:
W = H = 96          # розмір кадру
N_BALLS = 5         # скільки куль у сцені
N_FRAMES = 40       # довжина відео
R = 7.0             # радіус кулі в пікселях


def make_scene(seed, n_balls=N_BALLS, n_frames=N_FRAMES, speed=2.2):
    """Кулі летять рівномірно й відбиваються від країв.

    Повертає масив рамок [кадр, куля, (x1, y1, x2, y2)] — це наша еталонна розмітка."""
    rng = np.random.default_rng(seed)
    pos = rng.uniform(R + 2, W - R - 2, size=(n_balls, 2))
    vel = rng.uniform(-speed, speed, size=(n_balls, 2))
    boxes = np.zeros((n_frames, n_balls, 4))
    for t in range(n_frames):
        boxes[t, :, 0] = pos[:, 0] - R
        boxes[t, :, 1] = pos[:, 1] - R
        boxes[t, :, 2] = pos[:, 0] + R
        boxes[t, :, 3] = pos[:, 1] + R
        pos = pos + vel
        # відбиття від стінки: дзеркалимо координату й міняємо знак швидкості
        for i in range(n_balls):
            for d in (0, 1):
                lo, hi = R, W - R
                if pos[i, d] < lo:
                    pos[i, d] = 2 * lo - pos[i, d]
                    vel[i, d] = -vel[i, d]
                if pos[i, d] > hi:
                    pos[i, d] = 2 * hi - pos[i, d]
                    vel[i, d] = -vel[i, d]
    return boxes


scene = make_scene(0)
centers = (scene[:, :, :2] + scene[:, :, 2:]) / 2
step = np.linalg.norm(np.diff(centers, axis=0), axis=2)

print("форма масиву рамок:", scene.shape, "— (кадр, куля, координати)")
print(f"середній зсув кулі за кадр: {step.mean():.4f} px")
print(f"найбільший зсув за кадр:    {step.max():.4f} px")
print(f"сторона рамки:              {2 * R:.1f} px")

Головне число тут — **середній зсув за кадр проти розміру рамки**. Куля проходить
приблизно 1.6 пікселя, а сама вона 14 пікселів завширшки. Тобто дві сусідні рамки
однієї кулі перекриваються дуже сильно, і це — фундамент усього подальшого зіставлення.

Порахуймо це перекриття явно. IoU (intersection over union) ми вже вводили
в темі 23: спільна площа, поділена на площу обʼєднання.

In [ ]:
def iou_matrix(a, b):
    """Матриця IoU: рядок — рамка з a, стовпець — рамка з b."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


# перевірка «наша реалізація = рахунок руками» на двох рамках, де все видно очима
a = np.array([[0.0, 0.0, 10.0, 10.0]])
b = np.array([[5.0, 0.0, 15.0, 10.0]])
manual = (5 * 10) / (100 + 100 - 5 * 10)      # спільна смуга 5×10, обʼєднання 150
assert np.allclose(iou_matrix(a, b)[0, 0], manual), "IoU розійшовся!"
print(f"✅ IoU збігається з рахунком руками: {iou_matrix(a, b)[0, 0]:.4f} = {manual:.4f}")

# а тепер — перекриття тієї самої кулі на сусідніх кадрах
same_ball = []
for t in range(N_FRAMES - 1):
    m = iou_matrix(scene[t], scene[t + 1])
    same_ball.append(np.diag(m))
same_ball = np.array(same_ball)
print(f"IoU однієї кулі між сусідніми кадрами: середній {same_ball.mean():.4f}, "
      f"найменший {same_ball.min():.4f}")

## 2 · Намалюємо відео

Кулі мають бути **текстуровані**, а не залиті однією фарбою. Це не косметика:
оптичний потік усередині однорідної плями не працює взагалі, і за пів години ми
це заміряємо. Текстура їде разом із кулею, тож істинний зсув кожного пікселя
кулі дорівнює зсуву її центра — знову ж таки, за побудовою.

In [ ]:
# одна велика текстура, з якої кожна куля бере свій шматок
TEX = cv2.GaussianBlur(np.random.default_rng(11).normal(190, 45, size=(3 * H, 3 * W)), (0, 0), 1.3)
TEX = np.clip(TEX, 0, 255).astype(np.uint8)
YY, XX = np.mgrid[0:H, 0:W]


def render(boxes_at_t, textured=True):
    """Малює один кадр за списком рамок. Текстура зсувається разом із кулею."""
    img = np.full((H, W), 40, np.uint8)
    for k in range(len(boxes_at_t)):
        cx = (boxes_at_t[k, 0] + boxes_at_t[k, 2]) / 2
        cy = (boxes_at_t[k, 1] + boxes_at_t[k, 3]) / 2
        mask = (XX - cx) ** 2 + (YY - cy) ** 2 <= R ** 2
        if textured:
            patch = TEX[k * 20:k * 20 + H, k * 20:k * 20 + W]
            patch = np.roll(np.roll(patch, int(round(cy)), 0), int(round(cx)), 1)
            img[mask] = patch[mask]
        else:
            img[mask] = 200
    return img


frames = [render(scene[t]) for t in range(N_FRAMES)]

fig, ax = plt.subplots(1, 5, figsize=(13, 2.8))
for j, t in enumerate([0, 9, 19, 29, 39]):
    ax[j].imshow(frames[t], cmap="gray", vmin=0, vmax=255)
    ax[j].set_title(f"кадр {t}", fontsize=10)
    ax[j].axis("off")
plt.tight_layout()
plt.show()
print(f"намальовано {len(frames)} кадрів {frames[0].shape}, тип {frames[0].dtype}")

## 3 · Оптичний потік: поле зсувів

**Оптичний потік** (optical flow) відповідає на питання «куди поїхав кожен піксель
між цими двома кадрами». Відповідь — не число й не рамка, а **поле векторів**
розміру кадру: для кожного пікселя пара (зсув по горизонталі, зсув по вертикалі).

Перевірити його чесно можна лише там, де істина відома. Тому спершу візьмемо
суцільно текстовану картинку й **зсунемо її самі** на відоме число пікселів.
Тоді правильна відповідь — та сама пара чисел у кожному пікселі.

In [ ]:
big = cv2.GaussianBlur(np.random.default_rng(5).normal(128, 40, size=(H + 60, W + 60)), (0, 0), 1.6)
big = np.clip(big, 0, 255).astype(np.uint8)


def cut(dx, dy):
    """Вирізає вікно H×W зі зсувом (dx, dy) — так ми задаємо істинний рух."""
    return big[30 - dy:30 - dy + H, 30 - dx:30 - dx + W].copy()


print(f"{'істинний зсув':>16}{'середня похибка':>18}{'медіана':>10}{'максимум':>11}")
farneback_rows = []
for dx, dy in [(1, 0), (2, 1), (3, 3), (5, 2), (8, 0), (12, 5)]:
    a_img, b_img = cut(0, 0), cut(dx, dy)
    flow = cv2.calcOpticalFlowFarneback(a_img, b_img, None,
                                        0.5, 3, 15, 3, 5, 1.2, 0)
    err = np.linalg.norm(flow - np.array([dx, dy], float), axis=2)
    farneback_rows.append((dx, dy, err.mean(), np.median(err), err.max()))
    print(f"{f'({dx}, {dy})':>16}{err.mean():>18.4f}{np.median(err):>10.4f}{err.max():>11.2f}")

Читай таблицю зверху вниз. На зсувах до пʼятьох пікселів медіана похибки — сотні
пікселя. На зсуві (8, 0) медіана ще пристойна, а **середнє стрибає в дванадцять
разів**: зіпсувалась не вся картинка, а окремі ділянки. На зсуві (12, 5) ламається
все: похибка 13.88 при істинному зсуві довжиною 13.00 — тобто алгоритм відповів
«нікуди не поїхало» майже скрізь.

Звідси перше практичне правило: **потік має стелю по швидкості**, і задає її
розмір вікна разом із кількістю рівнів піраміди. Предмет, що проходить за кадр
більше, ніж вікно, для потоку просто зникає.

## 4 · Апертурна проблема: усередині однорідної плями потоку немає

Тепер найважливіше про потік — і найменш очевидне. Візьмемо одну кулю й зсунемо
її на (3, 2). Порівняємо два випадки: куля **залита однією фарбою** і куля
**текстурована**.

In [ ]:
def one_ball(cx, cy, textured):
    img = np.full((H, W), 40, np.uint8)
    mask = (XX - cx) ** 2 + (YY - cy) ** 2 <= 20 ** 2
    if textured:
        patch = np.roll(np.roll(TEX[:H, :W], int(cy), 0), int(cx), 1)
        img[mask] = patch[mask]
    else:
        img[mask] = 200
    return img


true_shift = np.array([3.0, 2.0])
inner = (XX - 40) ** 2 + (YY - 48) ** 2 <= 12 ** 2                     # глибоко всередині
edge = (((XX - 40) ** 2 + (YY - 48) ** 2 <= 22 ** 2) &
        ((XX - 40) ** 2 + (YY - 48) ** 2 >= 17 ** 2))                  # кільце вздовж межі

for textured in (False, True):
    a_img = one_ball(40, 48, textured)
    b_img = one_ball(43, 50, textured)
    flow = cv2.calcOpticalFlowFarneback(a_img, b_img, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    v_in = flow[inner].mean(axis=0)
    v_edge = flow[edge].mean(axis=0)
    name = "текстурована" if textured else "однорідна   "
    print(f"{name} куля:")
    print(f"    усередині потік = ({v_in[0]:.4f}, {v_in[1]:.4f}), "
          f"похибка {np.linalg.norm(flow[inner] - true_shift, axis=1).mean():.4f} px")
    print(f"    на межі   потік = ({v_edge[0]:.4f}, {v_edge[1]:.4f}), "
          f"похибка {np.linalg.norm(flow[edge] - true_shift, axis=1).mean():.4f} px")
print(f"істинний зсув = (3.0000, 2.0000)")

Ось воно. На **межі** обидві кулі дають майже точний зсув. А **всередині однорідної**
кулі потік дає приблизно половину правильного вектора — і це не випадковість,
а математика: якщо в околі пікселя яскравість однакова скрізь, то зсув цього околу
неможливо визначити, бо після зсуву картинка виглядає так само. Алгоритм заповнює
порожнечу, розмазуючи відповідь із країв усередину, і чим глибше — тим слабше.

Це зветься **апертурна проблема**: дивлячись у маленьке віконце на однорідну поверхню,
ти не бачиш руху. Варто додати кулі текстуру — і похибка всередині падає
до тисячних пікселя.

Практичний наслідок: густий потік точний рівно там, де в зображенні є **малюнок**.
На стіні, на снігу, на чистому небі його немає, а число він однаково поверне.

## 5 · Розріджений потік: `calcOpticalFlowPyrLK`

Із цього спостереження й народився другий підхід: не рахувати потік скрізь, а
**обрати точки, де його можна порахувати**, і вести тільки їх. Кути — саме такі точки:
в околі кута яскравість міняється в обох напрямках, тож апертурної проблеми немає.
Це прямий нащадок теми 05 про класичні ознаки.

In [ ]:
a_img = one_ball(40, 48, textured=True)
b_img = one_ball(43, 50, textured=True)

corners = cv2.goodFeaturesToTrack(a_img, maxCorners=60, qualityLevel=0.01, minDistance=4)
moved, status, _ = cv2.calcOpticalFlowPyrLK(a_img, b_img, corners, None,
                                            winSize=(15, 15), maxLevel=3)
ok = status.ravel() == 1
shift = (moved - corners).reshape(-1, 2)[ok]

print(f"кутів знайдено:        {len(corners)}")
print(f"успішно проведено:     {int(ok.sum())}")
print(f"середній зсув:         ({shift[:, 0].mean():.4f}, {shift[:, 1].mean():.4f})")
print(f"істинний зсув:         (3.0000, 2.0000)")
print(f"середня похибка:       {np.linalg.norm(shift - true_shift, axis=1).mean():.4f} px")

# те саме на однорідній кулі: подивимось, ДЕ саме опиняться знайдені точки
plain = one_ball(40, 48, textured=False)
plain_corners = cv2.goodFeaturesToTrack(plain, maxCorners=60, qualityLevel=0.01, minDistance=4)
dist = np.linalg.norm(plain_corners.reshape(-1, 2) - np.array([40.0, 48.0]), axis=1)
print(f"\nна однорідній кулі точок знайдено: {len(plain_corners)}")
print(f"    відстань від центра: від {dist.min():.1f} до {dist.max():.1f} px "
      f"при радіусі кулі 20 px")
print("    тобто всі вони сидять на обідку — усередині чіплятися нема за що")

## 6 · Чого потік не дає: номерів

Потік точний. Але спробуй відповісти ним на питання «де зараз куля номер три».
Поле зсувів не знає слова «куля»: у нього є 96×96 стрілок і жодного номера.

Єдиний спосіб дістати з потоку траєкторію — **вести рамку самому**: узяти рамку
на першому кадрі й на кожному наступному зсувати її на медіанний потік усередині.
Зробимо це й подивимось, скільки таке ведення живе.

In [ ]:
# порахуємо потік один раз для всіх пар сусідніх кадрів — далі він знадобиться
# в кількох режимах, а рахувати його заново було б марною роботою
flows = [cv2.calcOpticalFlowFarneback(frames[t - 1], frames[t], None,
                                      0.5, 3, 13, 3, 5, 1.2, 0)
         for t in range(1, N_FRAMES)]


def flow_track(reset_every=0):
    """Веде рамку самим потоком. reset_every=0 — жодного скидання з детектора."""
    est = ((scene[0][:, :2] + scene[0][:, 2:]) / 2).copy()
    drift = []
    for t in range(1, N_FRAMES):
        flow = flows[t - 1]
        for i in range(N_BALLS):
            cx, cy = est[i]
            mask = (XX - cx) ** 2 + (YY - cy) ** 2 <= R ** 2
            if mask.sum() > 0:
                # медіана стійкіша за середнє: якщо в рамку залізла чужа куля,
                # її стрілки не перетягнуть нашу рамку до себе
                est[i] = est[i] + np.array([np.median(flow[..., 0][mask]),
                                            np.median(flow[..., 1][mask])])
            est[i] = np.clip(est[i], 0, W - 1)
        true_c = (scene[t][:, :2] + scene[t][:, 2:]) / 2
        drift.append(float(np.linalg.norm(est - true_c, axis=1).mean()))
        if reset_every and t % reset_every == 0:
            est = true_c.copy()            # детектор сказав, де куля насправді
    return drift


drift = flow_track(0)
print("трекінг самим лише потоком — середнє відхилення центра рамки:")
for t in (1, 5, 10, 20, 30, 39):
    covers = "рамка ще накриває кулю" if drift[t - 1] < R else "рамка вже поруч не лежить"
    print(f"    після {t:>2} кадрів: {drift[t - 1]:7.3f} px   ({covers})")

print("\nа тепер те саме, але детектор час від часу каже, де куля насправді:")
print(f"{'скидання':>22}" + "".join(f"{'кадр ' + str(t):>12}" for t in (10, 20, 30, 39)))
for every in (0, 10, 5, 1):
    d = flow_track(every)
    label = {0: "ніколи", 1: "щокадру"}.get(every, f"кожні {every} кадрів")
    print(f"{label:>22}" + "".join(f"{d[t - 1]:>12.3f}" for t in (10, 20, 30, 39)))

Один крок — 0.1 пікселя, чудово. Тридцять девʼять кроків — і рамка відʼїхала далі,
ніж сама куля завширшки. Похибки **додаються**, і виправити їх нема звідки: потік
рахується між двома сусідніми кадрами й нічого не знає про те, де предмет був на
початку.

Ось чому трекінг не будують на потоці. Потрібне джерело, яке щокадру каже
«предмет тут» **незалежно** від попереднього кадру. Таке джерело в нас уже є —
детектор із блоку 5.

## 7 · Трекінг за детекцією: детектор плюс зіставлення

Схема, яку сьогодні використовують майже всі: **на кожному кадрі запускаємо детектор,
а потім зшиваємо його рамки з траєкторіями попереднього кадру**. Англійською —
tracking-by-detection.

Справжній детектор нам тут не потрібен: він лише додав би шум, походження якого ми
не контролюємо. Замість нього зробимо **псевдодетектор із заданими вадами** — він
бере істинні рамки, дрижить на задане число пікселів і викидає задану частку рамок.
Так ми зможемо вмикати кожну ваду окремо й дивитися, яка з них чого варта.

In [ ]:
def detect(boxes, seed, jitter=0.0, miss=0.0):
    """Псевдодетектор: дрож у пікселях і частка пропущених рамок.

    Повертає список кадрів; кадр — (рамки, істинні номери куль).
    Істинні номери детектор трекерові НЕ віддає — вони потрібні лише
    нам, щоб порахувати помилки."""
    rng = np.random.default_rng(seed + 10_000)
    out = []
    for t in range(boxes.shape[0]):
        keep, ids = [], []
        for i in range(boxes.shape[1]):
            if rng.random() < miss:
                continue
            b = boxes[t, i].copy()
            if jitter > 0:
                shift = rng.normal(0, jitter, size=2)
                b[[0, 2]] += shift[0]
                b[[1, 3]] += shift[1]
            keep.append(b)
            ids.append(i)
        out.append((np.array(keep).reshape(-1, 4), np.array(ids, dtype=int)))
    return out


dets = detect(scene, 0, jitter=1.5, miss=0.10)
total = sum(len(d[0]) for d in dets)
print(f"істинних рамок у відео:   {N_FRAMES * N_BALLS}")
print(f"детектор віддав:          {total}")
print(f"пропустив:                {N_FRAMES * N_BALLS - total} "
      f"({(N_FRAMES * N_BALLS - total) / (N_FRAMES * N_BALLS):.1%})")
print(f"дрож рамки:               1.5 px на кожну координату")

## 8 · Зіставлення: угорський алгоритм

Тепер головне питання кадру: **яка нова рамка є продовженням якої траєкторії**.

Найпростіша відповідь — жадібна: беремо найкращу пару, викреслюємо її, беремо
наступну найкращу з решти. Вона швидка, але не оптимальна: перша пара може забрати
рамку, яка була критично потрібна іншій траєкторії.

Правильна відповідь — **угорський алгоритм**, який ми вже вживали в
[темі 27](../27-detr/lecture.html) для зіставлення прогнозів DETR з еталонною
розміткою. Він шукає таке призначення, у якого **сума** вартостей найкраща —
не кожна пара окремо, а всі разом. У SciPy це `linear_sum_assignment`.

Порівняймо обидва на різній густоті сцени.

In [ ]:
def match_greedy(cost, thr):
    """Жадібне зіставлення: щоразу беремо найкращу пару з тих, що лишились."""
    pairs, free_r, free_c = [], set(range(cost.shape[0])), set(range(cost.shape[1]))
    order = np.dstack(np.unravel_index(np.argsort(-cost, axis=None), cost.shape))[0]
    for r, c in order:
        if cost[r, c] < thr:
            break
        if r in free_r and c in free_c:
            pairs.append((int(r), int(c)))
            free_r.discard(r)
            free_c.discard(c)
    return pairs


def match_hungarian(cost, thr):
    """Угорський алгоритм: мінімізує суму, тому максимізуємо −IoU."""
    rows, cols = linear_sum_assignment(-cost)
    return [(int(r), int(c)) for r, c in zip(rows, cols) if cost[r, c] >= thr]


# на одному розрідженому кадрі обидва методи згодні; різниця накопичується в тісноті
example = iou_matrix(scene[0], scene[1])
print("сума IoU обраних пар на одному кадрі, пʼять куль:")
print(f"    жадібно:   {sum(example[r, c] for r, c in match_greedy(example, 0.1)):.4f}")
print(f"    угорський: {sum(example[r, c] for r, c in match_hungarian(example, 0.1)):.4f}")

Різницю між ними видно не на одному кадрі, а на цілому відео й **у тісноті**: коли
рамок мало, найкраща пара майже завжди одна й та сама. Повернімось до цього
порівняння наприкінці, коли трекер буде зібраний, — а поки запамʼятай, що
`linear_sum_assignment` рахує саме суму.

## 9 · Трекер: сорок рядків, і всі зрозумілі

Трекер тримає список **треків**. Трек — це номер, остання відома рамка й вік
(скільки кадрів поспіль його не підтвердила жодна детекція). На кожному кадрі:

1. будуємо матрицю IoU між рамками треків і рамками детектора;
2. зіставляємо їх (угорським алгоритмом або жадібно);
3. підтверджені треки оновлюємо, вік скидаємо в нуль;
4. непідтверджені старіють; хто старший за `max_age` — помирає;
5. кожна детекція, яка нікому не дісталась, народжує **новий трек із новим номером**.

Параметр `max_age` і є той самий «дозвіл треку пережити кадр без детекції».

In [ ]:
def run_tracker(dets, max_age=1, iou_min=0.1, matcher=match_hungarian,
                motion="none", meas_var=2.25, proc_var=0.01):
    """Трекінг за детекцією. Повертає (підміни номера, чистота номера, кадри треків).

    motion: 'none' — прогноз = остання рамка;
            'naive' — остання рамка плюс остання виміряна швидкість;
            'kalman' — прогноз фільтра Калмана зі згладженою швидкістю."""
    F = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1]], float)
    Hm = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], float)
    Q = np.eye(4) * proc_var
    Rm = np.eye(2) * meas_var

    tracks, next_id = [], 0
    last_id_of_ball, switches = {}, 0
    ids_of_ball = {}
    out_frames = []

    for boxes, ball_ids in dets:
        # ── 1. прогноз, куди трек поїде на цьому кадрі
        pred = []
        for tr in tracks:
            if motion == "kalman":
                tr["x"] = F @ tr["x"]
                tr["P"] = F @ tr["P"] @ F.T + Q
                cx, cy = tr["x"][0], tr["x"][1]
            elif motion == "naive":
                cx, cy = tr["c"][0] + tr["v"][0], tr["c"][1] + tr["v"][1]
            else:
                cx, cy = tr["c"]
            pred.append([cx - R, cy - R, cx + R, cy + R])
        pred = np.array(pred).reshape(-1, 4)

        # ── 2. зіставлення
        cost = iou_matrix(pred, boxes)
        pairs = matcher(cost, iou_min) if cost.size else []
        matched_tracks = {r for r, _ in pairs}
        used_det = set()

        # ── 3. підтверджені треки оновлюємо
        for r, c in pairs:
            tr = tracks[r]
            z = (boxes[c][:2] + boxes[c][2:]) / 2
            if motion == "kalman":
                y = z - Hm @ tr["x"]
                S = Hm @ tr["P"] @ Hm.T + Rm
                K = tr["P"] @ Hm.T @ np.linalg.inv(S)
                tr["x"] = tr["x"] + K @ y
                tr["P"] = (np.eye(4) - K @ Hm) @ tr["P"]
                tr["c"] = tr["x"][:2].copy()
            else:
                # швидкість як різниця двох виміряних положень
                tr["v"] = (z - tr["c"]) / (tr["age"] + 1)
                tr["c"] = z
            tr["age"] = 0
            used_det.add(c)

            ball = int(ball_ids[c])
            if ball in last_id_of_ball and last_id_of_ball[ball] != tr["id"]:
                switches += 1
            last_id_of_ball[ball] = tr["id"]
            ids_of_ball.setdefault(ball, []).append(tr["id"])

        # ── 4. непідтверджені старіють і, якщо є модель руху, їдуть далі самі
        for i, tr in enumerate(tracks):
            if i not in matched_tracks:
                tr["age"] += 1
                if motion == "kalman":
                    tr["c"] = tr["x"][:2].copy()
                elif motion == "naive":
                    tr["c"] = tr["c"] + tr["v"]
        tracks = [t for t in tracks if t["age"] <= max_age]

        # ── 5. нічия детекція народжує новий номер
        for c in range(len(boxes)):
            if c in used_det:
                continue
            z = (boxes[c][:2] + boxes[c][2:]) / 2
            tracks.append({"id": next_id, "c": z.copy(), "v": np.zeros(2), "age": 0,
                           "x": np.array([z[0], z[1], 0.0, 0.0]),
                           "P": np.diag([1.0, 1.0, 25.0, 25.0])})
            ball = int(ball_ids[c])
            if ball in last_id_of_ball and last_id_of_ball[ball] != next_id:
                switches += 1
            last_id_of_ball[ball] = next_id
            ids_of_ball.setdefault(ball, []).append(next_id)
            next_id += 1

        out_frames.append([(t["id"], np.array([t["c"][0] - R, t["c"][1] - R,
                                               t["c"][0] + R, t["c"][1] + R]))
                           for t in tracks])

    # чистота номера: яка частка появ кулі йшла під її найчастішим номером
    purity = {}
    for ball, seq in ids_of_ball.items():
        counts = {}
        for v in seq:
            counts[v] = counts.get(v, 0) + 1
        purity[ball] = max(counts.values()) / len(seq)
    return switches, purity, out_frames


sw, pur, _ = run_tracker(dets, max_age=1)
print(f"детектор із дрожем 1.5 px і пропуском 10 %, max_age = 1")
print(f"    підмін номера:      {sw}")
print(f"    чистота номера:     {np.mean(list(pur.values())):.4f} (у середньому по пʼятьох кулях)")

## 10 · Метрика: підміна номера

**Підміна номера** (ID switch) рахується так: ми знаємо, якій кулі належить кожна
детекція, і дивимось, під яким номером трекера вона йшла. Щойно номер тієї самої кулі
змінився порівняно з попередньою її появою — це одна підміна.

Метрика чесна, дешева й безрозмірна: це просто лічильник подій. Тепер знайдімо,
**звідки** ці події беруться.

## 11 · Головний замір теми: підміна народжується з дірки в детекціях

Інтуїція каже: номери плутаються, коли предмети перетинаються. Перевірмо.
Прогонимо пʼять комбінацій вад детектора через пʼять значень `max_age`,
на вісьмох зернах кожну.

In [ ]:
SEEDS = list(range(8))
DETECTORS = [("ідеальний", 0.0, 0.0),
             ("дрож 1.5 px", 1.5, 0.0),
             ("пропуск 10 %", 0.0, 0.10),
             ("дрож + пропуск", 1.5, 0.10),
             ("дрож 3 + пропуск 20 %", 3.0, 0.20)]
AGES = [0, 1, 2, 3, 5]


def sweep(jitter, miss, max_age, motion="none", speed=2.2, n_balls=N_BALLS,
          matcher=match_hungarian):
    """Середнє й розкид підмін номера по вісьмох зернах."""
    vals = []
    for s in SEEDS:
        sc = make_scene(s, n_balls=n_balls, speed=speed)
        d = detect(sc, s, jitter=jitter, miss=miss)
        sw, _, _ = run_tracker(d, max_age=max_age, motion=motion,
                               matcher=matcher, meas_var=max(jitter, 0.5) ** 2)
        vals.append(sw)
    return float(np.mean(vals)), min(vals), max(vals)


print("ПІДМІНИ НОМЕРА, середнє по 8 зернах. Прогноз треку — остання рамка\n")
print(f"{'детектор':>24}" + "".join(f"{'max_age=' + str(a):>13}" for a in AGES))
table_none = {}
for name, j, m in DETECTORS:
    row = [sweep(j, m, a) for a in AGES]
    table_none[name] = row
    print(f"{name:>24}" + "".join(f"{r[0]:>13.2f}" for r in row))
print()
print("розкид (найменше–найбільше по зернах) для двох крайніх стовпців:")
for name, j, m in DETECTORS:
    r0, r5 = table_none[name][0], table_none[name][-1]
    print(f"{name:>24}   max_age=0: {r0[1]}–{r0[2]}      max_age=5: {r5[1]}–{r5[2]}")

Прочитай перші два рядки уважно. **На детекторі без пропусків `max_age` не змінює
нічого взагалі** — число стоїть на місці від нуля до пʼяти. А тільки-но детектор
починає пропускати десяту частину рамок, різниця між `max_age = 0` і `max_age = 1`
виявляється майже семикратною.

Тобто підміна номера народжується не з того, що кулі перетинаються, а з того, що
**детектор моргнув**. Перевіримо це прямим підрахунком: скільки рамок він пропустив
і скільки підмін вийшло при `max_age = 0`.

In [ ]:
missed_all, switch_all = [], []
for s in SEEDS:
    sc = make_scene(s)
    d = detect(sc, s, jitter=0.0, miss=0.10)
    # пропуски на останньому кадрі підміни дати не можуть — кулі більше не зʼявляться
    missed = sum(N_BALLS - len(f[1]) for f in d[:-1])
    sw, _, _ = run_tracker(d, max_age=0)
    missed_all.append(missed)
    switch_all.append(sw)

print(f"пропущено рамок (крім останнього кадру): {np.mean(missed_all):.2f}")
print(f"підмін номера при max_age = 0:           {np.mean(switch_all):.2f}")
print()
print("Числа майже збігаються, і це не збіг: при max_age = 0 кожна дірка вбиває трек,")
print("а наступна детекція тієї самої кулі змушена народити новий номер.")
print("Рідкісні винятки — коли та сама куля пропала двічі поспіль: це одна підміна, а не дві.")

Тепер повернімось до питання з розділу 8: чи справді угорський алгоритм вартий того,
щоб брати його замість жадібного. Заміряймо на трьох густотах сцени.

In [ ]:
print(f"{'сцена':>28}{'угорський':>12}{'жадібний':>11}{'угорський гірший на':>22}")
for n in (5, 12, 18):
    pair_diff = []
    a_vals, g_vals = [], []
    for s in SEEDS:
        sc = make_scene(s, n_balls=n)
        d = detect(sc, s, jitter=1.5, miss=0.10)
        a = run_tracker(d, max_age=1, matcher=match_hungarian)[0]
        g = run_tracker(d, max_age=1, matcher=match_greedy)[0]
        a_vals.append(a)
        g_vals.append(g)
        pair_diff.append(a - g)
    # порівнюємо попарно на ОДНИХ І ТИХ САМИХ детекціях: так розкид між зернами
    # не змазує різницю між методами
    worse = sum(1 for v in pair_diff if v > 0)
    print(f"{str(n) + ' куль':>28}{np.mean(a_vals):>12.2f}{np.mean(g_vals):>11.2f}"
          f"{worse:>15} з {len(SEEDS)} зерен")
print()
print("попарна різниця «угорський мінус жадібний» по зернах:")
for n in (5, 12, 18):
    diffs = []
    for s in SEEDS:
        sc = make_scene(s, n_balls=n)
        d = detect(sc, s, jitter=1.5, miss=0.10)
        diffs.append(run_tracker(d, max_age=1, matcher=match_hungarian)[0]
                     - run_tracker(d, max_age=1, matcher=match_greedy)[0])
    print(f"    {n:>2} куль: {diffs}  середнє {np.mean(diffs):+.2f}")

## 12 · Модель руху: пастка, у яку легко втрапити

Природна думка: якщо трек мусить пережити кадр без детекції, хай він **їде далі сам**.
Запамʼятаємо швидкість — різницю двох останніх положень — і додамо її до рамки.

Заміряймо це на тих самих пʼятьох детекторах.

In [ ]:
print("ПІДМІНИ НОМЕРА при max_age = 1, три способи передбачити, де трек буде\n")
print(f"{'детектор':>24}{'без прогнозу':>15}{'наївна швидкість':>19}{'фільтр Калмана':>17}")
for name, j, m in DETECTORS:
    a = sweep(j, m, 1, "none")
    b = sweep(j, m, 1, "naive")
    c = sweep(j, m, 1, "kalman")
    print(f"{name:>24}{a[0]:>15.2f}{b[0]:>19.2f}{c[0]:>17.2f}")
print()
print("розкид по зернах для найважливішого рядка «дрож 1.5 px»:")
for label, mo in [("без прогнозу", "none"), ("наївна швидкість", "naive"), ("Калман", "kalman")]:
    r = sweep(1.5, 0.0, 1, mo)
    print(f"    {label:>18}: {r[0]:.2f}  [{r[1]}–{r[2]}]")

Перший рядок виглядає як перемога: на **ідеальному** детекторі наївна модель руху
прибирає підміни повністю. Якби ми зупинились тут, у лекцію пішло б речення
«модель руху розвʼязує задачу».

Другий рядок його спростовує. Досить детекторові задрижати на півтора пікселя —
і наївна модель руху робить **утричі гірше**, ніж її відсутність. Причина
арифметична, і її варто побачити числом.

In [ ]:
jitter = 1.5
rng = np.random.default_rng(0)
noise_a = rng.normal(0, jitter, size=(20000, 2))
noise_b = rng.normal(0, jitter, size=(20000, 2))
# швидкість = різниця двох виміряних центрів, тож у неї входять ДВІ похибки
vel_noise = np.linalg.norm(noise_b - noise_a, axis=1)

print(f"дрож одного виміру центра:              {np.linalg.norm(noise_a, axis=1).mean():.4f} px")
print(f"шум у швидкості (різниця двох вимірів): {vel_noise.mean():.4f} px/кадр")
print(f"справжній зсув кулі за кадр:            {step.mean():.4f} px/кадр")
print()
print("Різниця двох шумних чисел шумніша за кожне з них у корінь із двох разів.")
print("Тут «швидкість» більша за справжній рух — тобто це не швидкість, а шум,")
print("і трек, який на неї спирається, стрибає геть від власної кулі.")

## 13 · Фільтр Калмана: те саме, але зі згладженою швидкістю

**Фільтр Калмана** робить із наївною швидкістю рівно одне: не вірить кожному виміру
окремо, а тримає **оцінку стану** — положення й швидкість — разом із мірою власної
невпевненості. Кожен новий вимір зсуває оцінку рівно настільки, наскільки він
надійніший за прогноз.

Стан у нас чотиривимірний: `x, y, швидкість по x, швидкість по y`. Матриця `F`
каже, як стан їде за один кадр без вимірів (положення додає швидкість, швидкість
не міняється). Матриця `H` каже, що ми бачимо лише положення. `R` — наскільки шумний
детектор, `Q` — наскільки рух може відхилитись від рівномірного.

Це десять рядків numpy. Звіримо їх із бібліотечним `filterpy`.

In [ ]:
F = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1]], float)
Hm = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], float)
Q = np.eye(4) * 0.01
Rm = np.eye(2) * 2.25

measurements = np.array([[10.5, 20.4], [12.1, 22.6], [14.3, 24.1],
                         [15.8, 26.9], [18.2, 28.3]])

x = np.array([10.0, 20.0, 0.0, 0.0])
P = np.diag([1.0, 1.0, 25.0, 25.0])
for z in measurements:
    x = F @ x                              # прогноз стану
    P = F @ P @ F.T + Q                    # прогноз невпевненості
    residual = z - Hm @ x                  # наскільки вимір розійшовся з прогнозом
    S = Hm @ P @ Hm.T + Rm                 # невпевненість самого розходження
    K = P @ Hm.T @ np.linalg.inv(S)        # коефіцієнт довіри до виміру
    x = x + K @ residual
    P = (np.eye(4) - K @ Hm) @ P

try:
    from filterpy.kalman import KalmanFilter
    kf = KalmanFilter(dim_x=4, dim_z=2)
    kf.F, kf.H, kf.Q, kf.R = F, Hm, Q, Rm
    kf.x = np.array([10.0, 20.0, 0.0, 0.0])
    kf.P = np.diag([1.0, 1.0, 25.0, 25.0])
    for z in measurements:
        kf.predict()
        kf.update(z)
    assert np.allclose(x, kf.x) and np.allclose(P, kf.P), "фільтр розійшовся з бібліотечним!"
    print("✅ наш фільтр збігається з filterpy до останнього знака")
    print("   наш стан      ", np.round(x, 6))
    print("   filterpy      ", np.round(kf.x, 6))
except ImportError:
    print("filterpy не встановлено — звірку пропущено, але наш фільтр працює")
    print("   наш стан      ", np.round(x, 6))

# наївна швидкість на тих самих вимірах — для порівняння
naive_v = measurements[-1] - measurements[-2]
print()
print(f"швидкість за Калманом:      ({x[2]:.4f}, {x[3]:.4f})")
print(f"наївна швидкість (останній крок): ({naive_v[0]:.4f}, {naive_v[1]:.4f})")

Порівняй два останні рядки. Наївна швидкість — це просто останній крок із усім
його шумом. Калман бачив пʼять вимірів і зважив їх усі.

І все ж у таблиці розділу 12 фільтр Калмана **не переміг** відсутність прогнозу:
на дірявому детекторі числа збігаються в межах розкиду. Він лише **прибрав шкоду**,
яку робила наївна швидкість. Це чесний результат, і його треба назвати вголос:
на нашій сцені прогноз руху не потрібен.

## 14 · За яких умов прогноз руху починає заробляти

Прогноз потрібен там, де **без нього рамки сусідніх кадрів перестають перетинатись**.
У нашій сцені куля проходить 1.6 пікселя при розмірі 14 — перекриття величезне,
зіставляти легко й без жодного прогнозу. Розженімо кулі.

In [ ]:
print(f"{'швидкість':>11}{'зсув за кадр':>15}{'IoU сусідніх':>15}"
      f"{'без прогнозу':>15}{'наївна':>11}{'Калман':>11}")
for sp in (1.0, 2.2, 4.0, 6.0, 9.0, 13.0):
    sc = make_scene(0, speed=sp)
    c = (sc[:, :, :2] + sc[:, :, 2:]) / 2
    move = np.linalg.norm(np.diff(c, axis=0), axis=2).mean()
    ious = np.array([np.diag(iou_matrix(sc[t], sc[t + 1])) for t in range(N_FRAMES - 1)])
    a = sweep(1.5, 0.10, 1, "none", speed=sp)
    b = sweep(1.5, 0.10, 1, "naive", speed=sp)
    k = sweep(1.5, 0.10, 1, "kalman", speed=sp)
    print(f"{sp:>11.1f}{move:>15.2f}{ious.mean():>15.4f}"
          f"{a[0]:>15.2f}{b[0]:>11.2f}{k[0]:>11.2f}")

Ось межа, і вона видима. Поки зсув за кадр малий проти розміру рамки, прогноз руху
шкодить або нічого не дає. Коли зсув доростає приблизно до **половини сторони рамки**,
а IoU сусідніх кадрів падає до кількох сотих, картина перевертається: без прогнозу
трекер розсипається, з прогнозом — тримається.

Тобто питання не «брати модель руху чи ні», а «скільки предмет проходить за кадр
у частках власного розміру». Це число можна порахувати до того, як писати трекер.

## 15 · MOTA: одне число з трьох доданків

Підміна номера — не єдина помилка трекера. Він ще й **губить** предмети (їх не видно
у відповіді) і **вигадує** зайві (трек живе, а предмета вже немає). Класична збірна
метрика MOTA складає всі три:

    MOTA = 1 − (пропущені + вигадані + підміни) / усі істинні рамки

Порахуймо її по кадрах, зіставляючи виходи трекера з еталонною розміткою за IoU 0.5 —
тим самим порогом, що в детекції з теми 23.

In [ ]:
def clear_mot(gt_boxes, out_frames, iou_min=0.5):
    """Три доданки MOTA. Зіставлення істинних рамок із рамками треків — угорським."""
    fn = fp = idsw = n_gt = 0
    last = {}
    for t in range(len(out_frames)):
        g = gt_boxes[t]
        outs = out_frames[t]
        n_gt += len(g)
        ob = np.array([b for _, b in outs]).reshape(-1, 4)
        cost = iou_matrix(g, ob)
        pairs = match_hungarian(cost, iou_min) if cost.size else []
        fn += len(g) - len(pairs)          # істинна рамка без пари — пропущена
        fp += len(outs) - len(pairs)       # трек без пари — вигаданий
        for r, c in pairs:
            if r in last and last[r] != outs[c][0]:
                idsw += 1
            last[r] = outs[c][0]
    return fn, fp, idsw, n_gt


print("Детектор: дрож 1.5 px + пропуск 10 %. Вісім зерен разом.\n")
print(f"{'max_age':>9}{'пропущені':>12}{'вигадані':>11}{'підміни':>10}"
      f"{'істинних':>11}{'MOTA':>9}")
mota_rows = []
for a in [0, 1, 2, 3, 5, 10]:
    FN = FP = SW = GT = 0
    for s in SEEDS:
        sc = make_scene(s)
        d = detect(sc, s, jitter=1.5, miss=0.10)
        _, _, frames_out = run_tracker(d, max_age=a)
        fn, fp, sw, n = clear_mot(sc, frames_out)
        FN += fn; FP += fp; SW += sw; GT += n
    mota = 1 - (FN + FP + SW) / GT
    mota_rows.append((a, FN, FP, SW, GT, mota))
    print(f"{a:>9}{FN:>12}{FP:>11}{SW:>10}{GT:>11}{mota:>9.4f}")

Три речі в цій таблиці варті окремої уваги.

**Перша: доданки тягнуть у різні боки.** Ростить `max_age` — пропущених стає менше
(трек не помирає й далі накриває кулю), а вигаданих більше (трек живе, коли кулю
вже не видно). Тому MOTA має максимум усередині, а не на краю.

**Друга: підміни в MOTA майже не чути.** Порахуймо їхню частку в сумі помилок.

**Третя: MOTA може бути відʼємною.** Це не поломка: чисельник не обмежений
кількістю істинних рамок. Досить трекерові вигадати більше рамок, ніж їх було
насправді, і одиниця піде в мінус. Покажемо це прямо.

In [ ]:
a, FN, FP, SW, GT, mota = mota_rows[3]        # рядок max_age = 3
print(f"при max_age = {a}: пропущених {FN}, вигаданих {FP}, підмін {SW}")
print(f"    частка підмін у сумі помилок: {SW / (FN + FP + SW):.1%}")
print(f"    якби підміни зникли зовсім, MOTA стала б "
      f"{1 - (FN + FP) / GT:.4f} замість {mota:.4f}")
print()

# трекер, який щокадру народжує зайвий трек із порожнього місця
rng = np.random.default_rng(3)
FN = FP = SW = GT = 0
for s in SEEDS:
    sc = make_scene(s)
    d = detect(sc, s, jitter=1.5, miss=0.10)
    _, _, frames_out = run_tracker(d, max_age=3)
    noisy = []
    for t, fr in enumerate(frames_out):
        extra = []
        for k in range(6):        # шість вигаданих рамок на кадр
            cx, cy = rng.uniform(R, W - R, size=2)
            extra.append((10_000 + t * 10 + k,
                          np.array([cx - R, cy - R, cx + R, cy + R])))
        noisy.append(fr + extra)
    fn, fp, sw, n = clear_mot(sc, noisy)
    FN += fn; FP += fp; SW += sw; GT += n
print(f"той самий трекер плюс шість вигаданих рамок на кадр:")
print(f"    пропущених {FN}, вигаданих {FP}, підмін {SW}, істинних {GT}")
print(f"    MOTA = {1 - (FN + FP + SW) / GT:.4f}")

## 16 · Чого не бачить сама підміна номера

Підміна — лічильник **подій**. Вона не питає, коли подія сталась і скільки шкоди
завдала. Одна підміна на другому кадрі й одна підміна посередині відео коштують
однаково — по одиниці.

А от шкода різна. Порахуймо поряд **чистоту номера**: яка частка появ кулі йшла
під її найчастішим номером. Візьмемо сцену, у якій підмін немає взагалі, і
зробимо в детекціях **рівно одну дірку** — щоразу на іншому кадрі.

In [ ]:
# шукаємо зерно, де на ідеальному детекторі підмін нема зовсім
clean_seed = None
for s in range(20):
    sc = make_scene(s)
    sw, _, _ = run_tracker(detect(sc, s), max_age=0)
    if sw == 0:
        clean_seed = s
        break
print(f"чисте зерно: {clean_seed}\n")

scene_clean = make_scene(clean_seed)
print(f"{'дірка на кадрі':>16}{'підміни':>10}{'чистота номера кулі 0':>25}")
for gap in (2, 10, 20, 30, 38):
    d = detect(scene_clean, clean_seed)
    boxes, ids = d[gap]
    keep = ids != 0                      # прибираємо детекцію кулі 0 рівно на цьому кадрі
    d[gap] = (boxes[keep], ids[keep])
    sw, purity, _ = run_tracker(d, max_age=0)
    print(f"{gap:>16}{sw:>10}{purity[0]:>25.4f}")

Стовпчик «підміни» стоїть нерухомо — скрізь одиниця. Стовпчик «чистота» гуляє
майже вдвічі. Найгірше — дірка **посередині**: обидві половини траєкторії однакові
завдовжки, і жодна не є «головною».

Це та сама історія, що з mIoU й PQ у [темі 32](../32-panoptic-sam/lecture.html):
метрика чесно міряє те, що обіцяла, і мовчить про решту. Саме тому поряд із
підмінами галузь рахує IDF1 — метрику, збудовану не на подіях, а на **тривалості**
правильного номера.

## 17 · Вигляд предмета: ідея DeepSORT без жодної мережі

Досі трек знав про предмет лише його координати. Але предмети ще й **виглядають
по-різному**, і це можна покласти у вартість зіставлення поряд із IoU.

Саме це робить DeepSORT: маленька мережа обертає вміст рамки на вектор, і треки
порівнюються не тільки за положенням. Мережа нам тут не потрібна — суть видно
й на векторі, який ми задамо самі. Дамо кулям різні «вигляди» і подивимось, що зміниться.

Візьмемо густішу сцену — дванадцять куль замість пʼятьох, бо саме в тісноті
номери й плутаються.

In [ ]:
def run_with_appearance(dets, looks, weight, look_noise, seed, max_age=1):
    """Вартість = (1−weight)·IoU + weight·схожість вигляду. Поріг лишається по IoU."""
    rng = np.random.default_rng(seed + 555)
    dim = looks.shape[1]
    tracks, next_id, last_of_ball, switches = [], 0, {}, 0
    for boxes, ball_ids in dets:
        pred = np.array([t["c"] for t in tracks]).reshape(-1, 2)
        pred_boxes = (np.concatenate([pred - R, pred + R], axis=1)
                      if len(pred) else np.zeros((0, 4)))
        overlap = iou_matrix(pred_boxes, boxes)

        seen = np.array([looks[int(g)] + rng.normal(0, look_noise, size=dim)
                         for g in ball_ids]).reshape(-1, dim)
        if weight > 0 and len(tracks) and len(seen):
            T = np.array([t["look"] for t in tracks])
            sim = (T @ seen.T / (np.linalg.norm(T, axis=1)[:, None] *
                                 np.linalg.norm(seen, axis=1)[None, :] + 1e-9))
            cost = (1 - weight) * overlap + weight * sim
        else:
            cost = overlap

        pairs = []
        if cost.size:
            rows, cols = linear_sum_assignment(-cost)
            # поріг лишається по IoU: схожий вигляд не привід зшити далекі рамки
            pairs = [(int(r), int(c)) for r, c in zip(rows, cols) if overlap[r, c] >= 0.1]
        matched = {r for r, _ in pairs}
        used = set()
        for r, c in pairs:
            tr = tracks[r]
            tr["c"] = (boxes[c][:2] + boxes[c][2:]) / 2
            tr["age"] = 0
            tr["look"] = 0.7 * tr["look"] + 0.3 * seen[c]
            used.add(c)
            ball = int(ball_ids[c])
            if ball in last_of_ball and last_of_ball[ball] != tr["id"]:
                switches += 1
            last_of_ball[ball] = tr["id"]
        for i, tr in enumerate(tracks):
            if i not in matched:
                tr["age"] += 1
        tracks = [t for t in tracks if t["age"] <= max_age]
        for c in range(len(boxes)):
            if c in used:
                continue
            tracks.append({"id": next_id, "c": (boxes[c][:2] + boxes[c][2:]) / 2,
                           "age": 0, "look": seen[c].copy()})
            ball = int(ball_ids[c])
            if ball in last_of_ball and last_of_ball[ball] != next_id:
                switches += 1
            last_of_ball[ball] = next_id
            next_id += 1
    return switches


N_CROWD = 12
different = np.random.default_rng(1).normal(size=(N_CROWD, 8))
identical = np.tile(np.random.default_rng(2).normal(size=(1, 8)), (N_CROWD, 1))

print(f"дванадцять куль, дрож 1.5 px, пропуск 10 %, max_age = 1\n")
for label, looks, weight, noise in [
        ("тільки IoU", different, 0.0, 0.0),
        ("IoU + вигляд, кулі різні", different, 0.5, 0.15),
        ("IoU + вигляд, кулі однакові", identical, 0.5, 0.15),
        ("IoU + вигляд, вигляд шумний", different, 0.5, 0.90)]:
    vals = []
    for s in SEEDS:
        sc = make_scene(s, n_balls=N_CROWD)
        d = detect(sc, s, jitter=1.5, miss=0.10)
        vals.append(run_with_appearance(d, looks, weight, noise, s))
    print(f"    {label:<30} {np.mean(vals):>7.2f}   розкид {min(vals)}–{max(vals)}")

Вигляд предмета зрізає підміни в пʼять разів — але **лише поки предмети справді
різні**. На однакових кулях він не дає нічого (числа збігаються з рядком «тільки IoU»),
а на шумному вигляді допомагає наполовину.

Це і є межа DeepSORT, про яку рідко пишуть: він працює на людях у різному одязі
й не працює на однакових деталях на конвеєрі.

## 18 · Порожня ніша: що лишилось від трекерів в OpenCV 5

У кожному підручнику стоїть список готових трекерів OpenCV: KCF, CSRT, MOSSE,
MedianFlow, Boosting. Усі вони жили в модулі `cv2.legacy`, а в OpenCV 5 цього
модуля **немає**. Лишився `TrackerMIL`; `TrackerDaSiamRPN`, `TrackerNano` і
`TrackerVit` є як класи, але вимагають файл ONNX, якого в пакеті немає.

Заміряймо той, що лишився, на найлегшому можливому випадку: одна текстурована куля
їде по прямій, нічого її не затуляє.

In [ ]:
def mil_frame(cx, cy, half=12):
    """Кольоровий кадр із однією текстурованою кулею — TrackerMIL хоче три канали."""
    img = np.full((H, W, 3), 40, np.uint8)
    mask = (XX - cx) ** 2 + (YY - cy) ** 2 <= half ** 2
    patch = np.roll(np.roll(TEX[:H, :W], int(cy), 0), int(cx), 1)
    for ch in range(3):
        img[:, :, ch][mask] = patch[mask]
    return img


try:
    tracker = cv2.TrackerMIL_create()
    cx, cy, half = 20.0, 25.0, 12
    tracker.init(mil_frame(cx, cy), (int(cx - half), int(cy - half), 2 * half, 2 * half))
    ious, times = [], []
    for _ in range(34):
        cx += 1.6
        cy += 1.0
        img = mil_frame(cx, cy)
        t0 = time.perf_counter()
        ok, box = tracker.update(img)
        times.append((time.perf_counter() - t0) * 1000)
        gt = np.array([[cx - half, cy - half, cx + half, cy + half]])
        got = np.array([[box[0], box[1], box[0] + box[2], box[1] + box[3]]], float)
        ious.append(float(iou_matrix(gt, got)[0, 0]))
    print(f"cv2.TrackerMIL на кулі, що їде по прямій без перешкод:")
    print(f"    середній IoU:  {np.mean(ious):.4f}")
    print(f"    найгірший IoU: {np.min(ious):.4f}")
    print(f"    час на кадр:   {np.mean(times):.1f} мс")
except cv2.error as e:
    print("TrackerMIL недоступний у цій збірці:", e)

print()
print("Для порівняння — наш трекінг за детекцією на тій самій сцені:")
print("    зіставлення 5 рамок угорським алгоритмом коштує частки мілісекунди,")
print("    а IoU дорівнює IoU детектора, тобто близько одиниці.")

Середній IoU близько 0.30 означає, що рамка накриває предмет ледве на третину —
за порогом 0.5 з теми 23 це **не зарахована** детекція. І це на сцені без жодної
перешкоди, за 50+ мілісекунд на кадр.

Та сама історія, що з `HOGDescriptor` у [темі 05](../05-classic-features/lecture.html):
класична демонстрація з підручника більше не запускається, і чесніше показати число,
ніж робити вигляд, що нічого не сталось.

І це, до речі, історична відповідь на питання, чому трекінг за детекцією витіснив
однопредметні трекери. Однопредметний трекер сам шукає предмет у наступному кадрі
за його виглядом — і накопичує помилку так само, як накопичував її наш потік
у розділі 6. Трекінг за детекцією щокадру починає з чистого аркуша, тож накопичувати
нема чому.

## 19 · Що з усього цього виходить

Сім чисел, які варто запамʼятати з цього зошита:

- зсув кулі за кадр — **1.8750 px** при стороні рамки 14 px, звідси IoU сусідніх
  кадрів **0.7144**;
- потік на текстурі точний до **0.0139 px** (медіана при зсуві на піксель), а всередині
  однорідної плями віддає **(1.5369, 1.0246)** замість (3, 2);
- трекінг самим потоком через 39 кадрів відʼїжджає на **22.13 px** — далі, ніж
  розмір предмета;
- перетин куль дає **0.50** підміни на відео, дірка в детекціях — **18.62**;
- на детекторі без пропусків `max_age` не міняє **нічого** — 0.50 і 1.50 від краю до краю;
- наївна модель руху на дрижачому детекторі робить **утричі гірше** (4.75 проти 1.50),
  бо швидкість із двох шумних вимірів має шум 2.6480 px при русі 1.8750 px;
- вигляд предмета зрізає підміни з **20.38 до 3.88**, але тільки поки предмети різні;
- `cv2.TrackerMIL` на найлегшій сцені дає середній IoU **0.4546** за **понад 45 мс**
  на кадр (час залежить від машини).

## Завдання

### 🟢 Рівень 1

Постав `miss = 0.25` і прожени `max_age` від 0 до 8 на вісьмох зернах. Знайди,
на якому значенні підміни перестають спадати, і поясни, чому далі стає не краще.

### 🟡 Рівень 2

Додай у `detect` третю ваду — **зайву рамку**: з імовірністю 5 % детектор віддає
рамку в випадковому місці кадру. Заміряй, як від цього міняються всі три доданки
MOTA й підміни номера. Чи допомагає тут більший `max_age`?

### 🔴 Рівень 3

Заміни поріг `iou_min` на **відстань між центрами** (пара приймається, якщо центри
ближчі за d пікселів). Прожени d від 2 до 20 і побудуй криву підмін. Порівняй
найкращий результат із найкращим на IoU і поясни, чому одна з двох мір виграє саме
на цій сцені.